In [0]:
agro_df = spark.table("silver.agrofood.agrofood_normalized_fin").dropDuplicates()
display(agro_df)

In [0]:
kca_df = spark.table("silver.kca_info.kca_normalized_fin").dropDuplicates()
display(kca_df)

In [0]:
kca_df = kca_df.withColumnRenamed("업태명", "업태")

In [0]:
df = agro_df.unionByName(kca_df)
display(df)

In [0]:
# from pyspark.sql.functions import col, count, countDistinct, sum as _sum, when, min as _min, max as _max, avg, stddev

# print(f"=== 기본 정보 ===")
# print(f"전체 행 수: {df.count():,}")
# print(f"카럼 수: {len(df.columns)}")
# print(f"카럼 목록: {df.columns}")

# # 카럼별 null 수 및 고유값 수
# print("\n=== 카럼별 null 수 / 고유값 수 ===")
# stats = df.select([
#     count(when(col(c).isNull(), c)).alias(f"{c}_null")
#     for c in df.columns
# ] + [
#     countDistinct(col(c)).alias(f"{c}_distinct")
#     for c in df.columns
# ]).collect()[0]

# for c in df.columns:
#     null_cnt = stats[f"{c}_null"]
#     dist_cnt = stats[f"{c}_distinct"]
#     print(f"  {c:10s} | null: {null_cnt:>10,} | 고유값: {dist_cnt:>8,}")

# # 숫자 카럼 기초통계 (가격, 단위_수치)
# print("\n=== 숫자 카럼 기초통계 ===")
# numeric_stats = df.select(
#     _min(col("가격").cast("double")).alias("가격_min"),
#     _max(col("가격").cast("double")).alias("가격_max"),
#     avg(col("가격").cast("double")).alias("가격_avg"),
#     stddev(col("가격").cast("double")).alias("가격_stddev"),
#     _min(col("단위_수치").cast("double")).alias("단위수치_min"),
#     _max(col("단위_수치").cast("double")).alias("단위수치_max"),
#     avg(col("단위_수치").cast("double")).alias("단위수치_avg"),
# ).collect()[0]

# print(f"  가격     | min: {numeric_stats['가격_min']:>12,.0f} | max: {numeric_stats['가격_max']:>12,.0f} | avg: {numeric_stats['가격_avg']:>12,.1f} | std: {numeric_stats['가격_stddev']:>12,.1f}")
# print(f"  단위_수치 | min: {numeric_stats['단위수치_min']:>12,.0f} | max: {numeric_stats['단위수치_max']:>12,.0f} | avg: {numeric_stats['단위수치_avg']:>12,.1f}")

# # 주요 카테고리 비율
# print("\n=== 카테고리별 행 수 ===")
# display(df.groupBy("카테고리").count().orderBy(col("count").desc()))

# print("\n=== 출처별 행 수 ===")
# display(df.groupBy("출처").count().orderBy(col("count").desc()))

In [0]:
# for c in df.columns:
#     total_cnt = df.filter(col(c).isNotNull()).count()
#     unique_cnt = df.select(col(c)).distinct().count()
#     print(f"{c:10s} | 전체값: {total_cnt:>10,} | 고유값: {unique_cnt:>8,}")

In [0]:
# display(df.groupBy("단위").count().orderBy(col("count").desc()))

In [0]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    "단위",
    when(col("단위") == "1리터", "1L").otherwise(col("단위"))
)

In [0]:
# display(df.select("카테고리","재료명", "세부속성", "단위", "등급").distinct())

In [0]:
# kca_sample = df.filter(col("출처") == "kca").limit(5000)
# agrofood_sample = df.filter(col("출처") == "agrofood").limit(5000)
# display(kca_sample.unionByName(agrofood_sample))

In [0]:
df = spark.table("silver.ingredient.ingredient").dropDuplicates()
display(df)

In [0]:
# 테이블 저장
df.dropDuplicates() \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.ingredient.ingredient")

In [0]:
# csv 저장
df.coalesce(1).write.mode("overwrite").option("header", "true").csv("/Volumes/silver/ingredient/ingredient")